# Démo 4 · Quelle est notre certitude ? Intervalles de confiance par bootstrap

Dans le [notebook 01](01_validation_and_hyperparameter_search.ipynb), la validation croisée
nous a donné un score moyen et un écart-type. Annoncer une accuracy de 0,70
laisse pourtant une question en suspens : **de combien ce nombre changerait-il avec d’autres
données ?** Un **intervalle de confiance** y répond par une plage de valeurs plausibles.

Pour beaucoup de quantités, il n’existe pas de formule simple pour cette plage. Le **bootstrap**
l’obtient par simulation : il construit de nombreux nouveaux jeux de données en
rééchantillonnant les données dont nous disposons, recalcule la quantité sur chacun et
observe à quel point elle varie.

À la fin, vous serez capables de :

- Expliquer ce que fait un **tirage avec remise** et ce que sont les observations **hors sac**
  (*out-of-bag*, OOB).
- Calculer un intervalle de confiance à 95 % par la méthode des centiles pour l’accuracy
  d’un classifieur.
- Visualiser l’incertitude d’une droite de régression et calculer un intervalle de confiance
  pour sa pente.
- Comparer un intervalle bootstrap avec la formule classique de `scipy.stats.linregress`.

Vous réutiliserez cette technique dans l’exercice Q6 du notebook
[03 · Titanic, de bout en bout](03_titanic_end_to_end.ipynb).

## Configuration

In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "numpy>=1.26",
        "pandas>=2.2,<4",
        "matplotlib>=3.7",
        "scikit-learn>=1.4",
        "scipy>=1.11",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import linregress

from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

### D’où viennent les données
Les fichiers CSV sont fournis avec le dépôt dans `demo_4/datasets/`. En local, la cellule
suivante remonte depuis le dossier du notebook jusqu’à trouver ce répertoire : aucun
téléchargement n’est nécessaire. Dans Colab, le dépôt n’est pas sur le disque; les mêmes
fichiers sont donc lus directement depuis GitHub.

In [ ]:
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/milarobotlearningcourse/data_science/main/demo_4/datasets/"

# Locally, find demo_4/datasets by walking up from the notebook folder.
ROOT, DATA_DIR = Path.cwd(), None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "demo_4" / "datasets").is_dir():
        ROOT, DATA_DIR = folder, folder / "demo_4" / "datasets"
        break


def data_source(filename):
    """Return the local CSV path if the repository is available, otherwise its URL."""
    if DATA_DIR is not None:
        return DATA_DIR / filename
    return DATA_URL + filename


print("Reading data from:", DATA_DIR or DATA_URL)

## A. Tirage avec remise

Un échantillon bootstrap a la **même taille** que les données d’origine, mais les lignes sont
tirées **avec remise** : certaines lignes apparaissent plusieurs fois, d’autres pas du tout.
Les lignes qui n’ont pas été tirées sont appelées lignes **hors sac** (OOB).

`rng` est un générateur aléatoire NumPy. Fixer sa graine (*seed*) rend tout le notebook reproductible.

In [ ]:
rng = np.random.default_rng(0)

rows = np.arange(10)
sample = rng.choice(rows, size=len(rows), replace=True)
out_of_bag = np.setdiff1d(rows, sample)

print("bootstrap sample:", np.sort(sample))
print("out-of-bag rows: ", out_of_bag)

En moyenne, un échantillon bootstrap contient environ **63 %** des lignes distinctes ; les
37 % restants sont hors sac. Comme le modèle ne voit jamais les lignes OOB pendant
l’entraînement, elles peuvent servir de jeu de test.

## B. Un intervalle de confiance pour l’accuracy d’un classifieur

### Les données : diabète des Indiens Pima
768 patients, 8 mesures médicales, et l’indication que la personne a développé ou non un
diabète dans les cinq ans (`Outcome` : 1 = oui). Le fichier n’a pas de ligne d’en-tête :
nous donnons donc nous-mêmes les noms des colonnes.

In [ ]:
columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome",
]
diabetes = pd.read_csv(data_source("diabetes.csv"), header=None, names=columns)
print(diabetes.shape)
display(diabetes.head())
print("share of positive cases:", round(diabetes["Outcome"].mean(), 3))

Environ 35 % des patients sont positifs : un modèle qui prédit toujours « pas de diabète »
atteindrait donc déjà environ 65 % d’accuracy. Gardez cette baseline en tête en lisant
les résultats.

### La boucle bootstrap
Pour chacune des 1 000 répétitions :

1. tirer un échantillon bootstrap de positions de lignes (avec remise) ;
2. entraîner un arbre de décision sur ces lignes ;
3. mesurer son accuracy sur les lignes hors sac.

Cela prend quelques secondes.

In [ ]:
X = diabetes.drop(columns="Outcome").to_numpy()
y = diabetes["Outcome"].to_numpy()
n = len(y)

n_iterations = 1000
scores = []
for _ in range(n_iterations):
    train_idx = rng.choice(n, size=n, replace=True)
    oob_idx = np.setdiff1d(np.arange(n), train_idx)

    model = DecisionTreeClassifier(random_state=0)
    model.fit(X[train_idx], y[train_idx])
    scores.append(accuracy_score(y[oob_idx], model.predict(X[oob_idx])))

scores = np.array(scores)
print(f"mean OOB accuracy over {n_iterations} bootstrap samples: {scores.mean():.3f}")

### De 1 000 scores à un intervalle
La **méthode des centiles** conserve les 95 % centraux des scores bootstrap : on retire
2,5 % de chaque queue, c’est-à-dire qu’on prend les 2,5e et 97,5e centiles.

In [ ]:
confidence = 95
tail = (100 - confidence) / 2  # percentage cut from each side

lower = np.percentile(scores, tail)
upper = np.percentile(scores, 100 - tail)
print(f"{confidence}% confidence interval for accuracy: [{lower:.1%}, {upper:.1%}]")

plt.hist(scores, bins=30, color="lightgray", edgecolor="gray")
plt.axvline(lower, linestyle="--", color="black", label=f"{tail}th percentile")
plt.axvline(upper, linestyle="-", color="black", label=f"{100 - tail}th percentile")
plt.xlabel("Out-of-bag accuracy")
plt.ylabel("Number of bootstrap samples")
plt.legend()
plt.show()

**Lecture du résultat.** L’intervalle s’étend d’environ 65 % à environ 75 % : une largeur
d’une dizaine de points. Un autre jeu de données de même taille pourrait facilement donner
un score supérieur ou inférieur de plusieurs points, et la borne inférieure de l’intervalle
ne fait pas mieux que la baseline de 65 %. Deux modèles dont les scores diffèrent de moins
que cette largeur ne devraient pas être qualifiés de « meilleur » ou de « moins bon » sans
preuves supplémentaires.

*Plus précisément :* cet intervalle décrit à quel point varie l’accuracy d’« un arbre de
décision entraîné sur des données comme les nôtres ». Il tient compte du fait que les lignes
d’entraînement et les lignes de test changent toutes les deux.

## C. L’incertitude d’une droite de régression

### Les données : score SAT et moyenne cumulative à l’université
84 étudiants : leur score au SAT (un test d’admission américain) et leur moyenne cumulative
(GPA) à l’université. Un score SAT plus élevé prédit-il une moyenne plus élevée, et avec
quelle précision connaissons-nous la pente ?

In [ ]:
sat_gpa = pd.read_csv(data_source("sat-gpa.csv"))
print(sat_gpa.shape)
display(sat_gpa.head())

X_sat = sat_gpa[["SAT"]].to_numpy()  # 2-D: one column, as scikit-learn expects
y_gpa = sat_gpa["GPA"].to_numpy()

Nous ajustons une droite sur 1 000 échantillons bootstrap, traçons chaque droite en gris
clair et conservons chaque pente. La droite rouge est l’ajustement sur l’ensemble des données.

In [ ]:
n_iterations = 1000
slopes = []
plt.figure(figsize=(10, 6))

for _ in range(n_iterations):
    idx = rng.choice(len(y_gpa), size=len(y_gpa), replace=True)
    lr = LinearRegression().fit(X_sat[idx], y_gpa[idx])
    slopes.append(lr.coef_[0])
    plt.plot(X_sat, lr.predict(X_sat), color="grey", alpha=0.05, zorder=1)

full_fit = LinearRegression().fit(X_sat, y_gpa)
plt.scatter(X_sat, y_gpa, color="orange", zorder=3, label="students")
plt.plot(X_sat, full_fit.predict(X_sat), color="red", zorder=4, label="fit on all the data")
plt.xlabel("SAT score")
plt.ylabel("University GPA")
plt.legend()
plt.show()

slopes = np.array(slopes)
print(f"slope on all the data: {full_fit.coef_[0]:.5f} GPA points per SAT point")
print(f"intercept: {full_fit.intercept_:.3f}")

La bande grise est étroite au milieu des données et plus large aux extrémités : les droites
peuvent pivoter autour du centre, si bien que les prédictions pour des scores SAT extrêmes
sont moins sûres.

### Un intervalle de confiance pour la pente
Même méthode des centiles qu’auparavant, appliquée aux 1 000 pentes.

In [ ]:
slope_low, slope_high = np.percentile(slopes, [2.5, 97.5])
print(f"bootstrap 95% CI for the slope: [{slope_low:.5f}, {slope_high:.5f}]")

plt.hist(slopes, bins=30, color="lightgray", edgecolor="gray")
plt.axvline(slope_low, linestyle="--", color="black")
plt.axvline(slope_high, linestyle="--", color="black")
plt.xlabel("Bootstrap slope")
plt.ylabel("Number of bootstrap samples")
plt.show()

### Comparaison avec la formule classique
Pour une droite, la statistique propose aussi une formule. `linregress` renvoie la pente, le
coefficient de corrélation `r`, une valeur p pour l’hypothèse « la vraie pente vaut 0 » et
l’**erreur type** de la pente. Un intervalle à 95 % vaut environ pente ± 1,96 × erreur type.

In [ ]:
fit = linregress(sat_gpa["SAT"], sat_gpa["GPA"])

print(f"slope:          {fit.slope:.5f}")
print(f"correlation r:  {fit.rvalue:.3f}")
print(f"p-value:        {fit.pvalue:.2e}")
print(f"standard error: {fit.stderr:.5f}")
print(f"classical 95% CI: [{fit.slope - 1.96 * fit.stderr:.5f}, {fit.slope + 1.96 * fit.stderr:.5f}]")

Les deux intervalles sont proches. La formule classique repose sur des hypothèses (par
exemple, des erreurs de loi normale avec une dispersion constante) ; le bootstrap suppose
seulement que nos 84 étudiants sont représentatifs de la population. Lorsqu’une formule
existe et que ses hypothèses sont respectées, utilisez-la. Lorsqu’il n’en existe pas (une
accuracy, un F1 score, une médiane…), le bootstrap fonctionne quand même.

L’intervalle reste bien à l’écart de 0 et la valeur p est minuscule : des scores SAT plus
élevés sont clairement associés à des moyennes plus élevées, d’environ 0,0017 point de GPA
par point de SAT (soit environ 0,17 point de GPA par tranche de 100 points de SAT).

### À vous
Calculez un intervalle de confiance bootstrap à **90 %** pour l’**ordonnée à l’origine** de
la régression. Vous devrez conserver `lr.intercept_` dans une nouvelle boucle bootstrap.
L’intervalle à 90 % est-il plus large ou plus étroit que ne le serait un intervalle à 95 % ?
Pourquoi ?

In [ ]:
# Your turn: bootstrap the intercept and compute a 90% interval.

<details>
<summary>Réponse de référence — à ouvrir après avoir essayé</summary>

```python
intercepts = []
for _ in range(1000):
    idx = rng.choice(len(y_gpa), size=len(y_gpa), replace=True)
    lr = LinearRegression().fit(X_sat[idx], y_gpa[idx])
    intercepts.append(lr.intercept_)

low, high = np.percentile(intercepts, [5, 95])
print(f"90% CI for the intercept: [{low:.3f}, {high:.3f}]")
```

Un intervalle à 90 % conserve les 90 % centraux des valeurs (on retire 5 % de chaque queue) :
il est donc **plus étroit** qu’un intervalle à 95 %. Nous acceptons de nous tromper plus
souvent en échange d’une plage plus resserrée. L’ordonnée à l’origine est la moyenne prédite
pour un score SAT de 0, très loin des données : c’est pourquoi son intervalle est large, et
elle n’a ici aucun sens pratique.

</details>

## À retenir

- Le bootstrap rééchantillonne les données **avec remise** pour simuler de nouveaux jeux de
  données de même taille.
- Dans chaque échantillon, environ 37 % des lignes sont **hors sac** et peuvent servir de jeu
  de test.
- La **méthode des centiles** transforme de nombreuses valeurs bootstrap en intervalle de
  confiance.
- Elle fonctionne pour n’importe quelle statistique, y compris celles sans formule, comme
  l’accuracy d’un classifieur.

**Suite :** [03 · Titanic, de bout en bout](03_titanic_end_to_end.ipynb) réunit les
notebooks 01 et 02 dans un projet complet.

**Pour aller plus loin :** la documentation scikit-learn de
[`resample`](https://scikit-learn.org/stable/modules/generated/sklearn.utils.resample.html),
une fonction utilitaire qui tire des échantillons bootstrap de tableaux ou de DataFrames.